# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 10.3 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference

In [5]:
TASK_ID = "task202"
CH = 10
H = W = 30
LOCAL_TASK_JSON = Path("/mnt/data/task202.json")
KAGGLE_TASK_JSON = Path(COMPETITION) / "task202.json"
TASK_JSON = LOCAL_TASK_JSON if LOCAL_TASK_JSON.exists() else KAGGLE_TASK_JSON
OUT_DIR = Path.cwd() / "task202_defect_propagation_onnx"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ONNX_PATH = OUT_DIR / f"{TASK_ID}.onnx"
SUBMISSION_PATH = Path.cwd() / "submission.zip"
SUMMARY_PATH = OUT_DIR / f"{TASK_ID}_validation_summary.json"

with TASK_JSON.open("r") as f:
    task = json.load(f)

print(TASK_ID, "train", len(task.get("train", [])), "test", len(task.get("test", [])), "arc-gen", len(task.get("arc-gen", [])))

task202 train 4 test 1 arc-gen 262


In [6]:

def grid_to_tensor(grid, h=H, w=W, ch=CH):
    """Encode a raw ARC grid into zero-padded [1,10,30,30].

    Important: only true in-grid color-0 pixels set channel 0.
    Padding remains all-zero across all channels.
    """
    gh, gw = len(grid), len(grid[0])
    assert gh <= h and gw <= w, f"grid {gh}x{gw} exceeds static {h}x{w} ONNX contract"
    x = np.zeros((1, ch, h, w), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, v in enumerate(row):
            x[0, int(v), r, c] = 1.0
    return x

def symbolic_rule_grid(grid):
    """Shape-generic reference implementation used for arc-gen diagnostics."""
    a = np.array(grid, dtype=np.int64)
    gh, gw = a.shape
    out = a.copy()

    for k in range(1, 10):
        coords = np.argwhere(a == k)
        if len(coords) == 0:
            continue

        rows = np.unique(coords[:, 0])
        cols = np.unique(coords[:, 1])
        row_region = np.zeros(gh, dtype=bool); row_region[rows] = True
        col_region = np.zeros(gw, dtype=bool); col_region[cols] = True

        defects = np.argwhere((a == 0) & row_region[:, None] & col_region[None, :])
        if len(defects) == 0:
            continue

        hcnt, wcnt = len(rows), len(cols)
        span_w = (wcnt == gw)
        span_h = (hcnt == gh)
        if span_w and not span_h:
            use_vertical_cut = True
        elif span_h and not span_w:
            use_vertical_cut = False
        else:
            use_vertical_cut = (wcnt >= hcnt)

        for r, c in defects:
            if use_vertical_cut:
                out[row_region, c] = 0
            else:
                out[r, col_region] = 0

    return out.tolist()

class Task202DefectPropagation(nn.Module):
    """Static symbolic ARC model.

    Input/output: [1,10,30,30].
    No tree model, no visible-template memorization, no dynamic shape export.
    """
    def __init__(self):
        super().__init__()

    def forward(self, x):
        active = (x.sum(dim=1, keepdim=True) > 0.5).float()

        row_act = (active.sum(dim=3, keepdim=True) > 0.5).float()
        col_act = (active.sum(dim=2, keepdim=True) > 0.5).float()
        h_act = row_act.sum(dim=2, keepdim=True)
        w_act = col_act.sum(dim=3, keepdim=True)

        zero_fill = torch.zeros_like(active)
        color_outputs = []

        for k in range(1, 10):
            mk = x[:, k:k+1]

            row_has = (mk.sum(dim=3, keepdim=True) > 0.5).float()
            col_has = (mk.sum(dim=2, keepdim=True) > 0.5).float()
            hcnt = row_has.sum(dim=2, keepdim=True)
            wcnt = col_has.sum(dim=3, keepdim=True)

            # Full same-color rectangular stripe, including zero defects inside it.
            region = row_has * col_has * active

            defects = x[:, 0:1] * region
            defect_cols = (defects.sum(dim=2, keepdim=True) > 0.5).float()
            defect_rows = (defects.sum(dim=3, keepdim=True) > 0.5).float()

            span_w = (torch.abs(wcnt - w_act) < 0.5).float()
            span_h = (torch.abs(hcnt - h_act) < 0.5).float()
            aspect_col = (wcnt >= hcnt).float()

            # Horizontal stripe -> vertical cut. Vertical stripe -> horizontal cut.
            # Square stripe tie is resolved by whether it spans the full active width/height;
            # if still ambiguous, use aspect_col.
            use_vertical_cut = torch.clamp(
                span_w * (1.0 - span_h)
                + span_w * span_h * aspect_col
                + (1.0 - span_w) * (1.0 - span_h) * aspect_col,
                0.0, 1.0
            )

            cut = region * torch.clamp(
                use_vertical_cut * defect_cols + (1.0 - use_vertical_cut) * defect_rows,
                0.0, 1.0
            )

            zero_fill = torch.clamp(zero_fill + cut, 0.0, 1.0)
            color_outputs.append(mk * (1.0 - cut) * active)

        zero_channel = torch.clamp(x[:, 0:1] + zero_fill, 0.0, 1.0) * active
        return torch.cat([zero_channel] + color_outputs, dim=1) * active

model = Task202DefectPropagation().eval()
print(model.__class__.__name__)


Task202DefectPropagation


In [7]:
dummy = torch.from_numpy(grid_to_tensor(task["test"][0]["input"]))

torch.onnx.export(
    model,
    dummy,
    str(ONNX_PATH),
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    do_constant_folding=True,
    dynamic_axes=None,
    dynamo=False,
)

onnx_model = onnx.load(str(ONNX_PATH))
onnx_model = onnx.shape_inference.infer_shapes(onnx_model)
onnx.save(onnx_model, str(ONNX_PATH))
onnx.checker.check_model(str(ONNX_PATH))
print("ONNX:", ONNX_PATH, "bytes:", ONNX_PATH.stat().st_size)

/tmp/ipykernel_16/4191058833.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX: /kaggle/working/task202_defect_propagation_onnx/task202.onnx bytes: 82500


In [8]:
def vi_shape(vi):
    return [int(d.dim_value) if d.dim_value else (d.dim_param or None) for d in vi.type.tensor_type.shape.dim]

onnx_model = onnx.load(str(ONNX_PATH))
ops = collections.Counter(node.op_type for node in onnx_model.graph.node)
forbidden = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}

summary_static = {
    "input_shape": vi_shape(onnx_model.graph.input[0]),
    "output_shape": vi_shape(onnx_model.graph.output[0]),
    "onnx_size_bytes": ONNX_PATH.stat().st_size,
    "ops": dict(ops),
    "forbidden_ops": sorted(forbidden & set(ops)),
    "function_count": len(getattr(onnx_model, "functions", [])),
}
print(json.dumps(summary_static, indent=2)[:3000])

assert summary_static["input_shape"] == [1, 10, 30, 30]
assert summary_static["output_shape"] == [1, 10, 30, 30]
assert summary_static["onnx_size_bytes"] < 1_400_000
assert not summary_static["forbidden_ops"]
assert summary_static["function_count"] == 0

{
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "onnx_size_bytes": 82500,
  "ops": {
    "Constant": 193,
    "ReduceSum": 59,
    "Greater": 39,
    "Cast": 66,
    "Slice": 10,
    "Mul": 119,
    "Sub": 54,
    "Abs": 18,
    "Less": 18,
    "GreaterOrEqual": 9,
    "Add": 37,
    "Clip": 28,
    "Concat": 1
  },
  "forbidden_ops": [],
  "function_count": 0
}


In [9]:

sess_options = ort.SessionOptions()
sess_options.intra_op_num_threads = 1
sess_options.inter_op_num_threads = 1
sess = ort.InferenceSession(str(ONNX_PATH), sess_options=sess_options, providers=["CPUExecutionProvider"])

def validate_onnx_examples(examples):
    ok = 0
    bad = []
    outside_zero_ok = 0

    for i, ex in enumerate(examples):
        gh, gw = len(ex["input"]), len(ex["input"][0])
        if gh > H or gw > W:
            bad.append({"index": i, "reason": f"raw grid {gh}x{gw} exceeds static {H}x{W}"})
            continue

        x = grid_to_tensor(ex["input"])
        y = sess.run(None, {"input": x})[0]
        pred = (y > 0.5).astype(np.float32)
        exp = grid_to_tensor(ex["output"])

        if np.array_equal(pred, exp):
            ok += 1
        else:
            bad.append({
                "index": i,
                "reason": "tensor_exact_mismatch",
                "diff_count": int(np.sum(pred != exp)),
            })

        active = (x.sum(axis=1, keepdims=True) > 0.5).astype(np.float32)
        outside_zero_ok += int(np.all(pred * (1.0 - active) == 0.0))

    return {
        "ok": ok,
        "total": len(examples),
        "bad_first10": bad[:10],
        "outside_zero_ok": outside_zero_ok,
    }

def validate_symbolic_examples(examples):
    ok = 0
    bad = []
    for i, ex in enumerate(examples):
        pred = np.array(symbolic_rule_grid(ex["input"]), dtype=np.int64)
        exp = np.array(ex["output"], dtype=np.int64)
        if np.array_equal(pred, exp):
            ok += 1
        else:
            bad.append({"index": i, "diff_count": int(np.sum(pred != exp))})
    return {"ok": ok, "total": len(examples), "bad_first10": bad[:10]}

arc_gen = task.get("arc-gen", [])
compatible_arc_gen = [ex for ex in arc_gen if len(ex["input"]) <= H and len(ex["input"][0]) <= W]
excluded_arc_gen = len(arc_gen) - len(compatible_arc_gen)

rng = random.Random(0)
inds = list(range(len(compatible_arc_gen)))
rng.shuffle(inds)
holdout_n = math.ceil(0.60 * len(compatible_arc_gen)) if compatible_arc_gen else 0
holdout = [compatible_arc_gen[i] for i in inds[:holdout_n]]

validation = {
    "strict_zero_padded_tensor_exact": {
        "train": validate_onnx_examples(task["train"]),
        "test": validate_onnx_examples(task["test"]),
        "arc_gen_60pct_compatible_holdout": validate_onnx_examples(holdout) if holdout else None,
        "arc_gen_compatible_count": len(compatible_arc_gen),
        "arc_gen_excluded_over_static_30x30": excluded_arc_gen,
    },
    "shape_generic_symbolic_diagnostic": {
        "arc_gen_all": validate_symbolic_examples(arc_gen) if arc_gen else None,
    },
}

summary = {
    "task_id": TASK_ID,
    "model_class": model.__class__.__name__,
    **summary_static,
    "validation": validation,
}
json.dump(summary, open(SUMMARY_PATH, "w"), indent=2)
print(json.dumps(summary, indent=2)[:5000])

assert validation["strict_zero_padded_tensor_exact"]["train"]["ok"] == validation["strict_zero_padded_tensor_exact"]["train"]["total"]
assert validation["strict_zero_padded_tensor_exact"]["test"]["ok"] == validation["strict_zero_padded_tensor_exact"]["test"]["total"]
if holdout:
    assert validation["strict_zero_padded_tensor_exact"]["arc_gen_60pct_compatible_holdout"]["ok"] == holdout_n
if arc_gen:
    assert validation["shape_generic_symbolic_diagnostic"]["arc_gen_all"]["ok"] == len(arc_gen)


{
  "task_id": "task202",
  "model_class": "Task202DefectPropagation",
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "onnx_size_bytes": 82500,
  "ops": {
    "Constant": 193,
    "ReduceSum": 59,
    "Greater": 39,
    "Cast": 66,
    "Slice": 10,
    "Mul": 119,
    "Sub": 54,
    "Abs": 18,
    "Less": 18,
    "GreaterOrEqual": 9,
    "Add": 37,
    "Clip": 28,
    "Concat": 1
  },
  "forbidden_ops": [],
  "function_count": 0,
  "validation": {
    "strict_zero_padded_tensor_exact": {
      "train": {
        "ok": 4,
        "total": 4,
        "bad_first10": [],
        "outside_zero_ok": 4
      },
      "test": {
        "ok": 1,
        "total": 1,
        "bad_first10": [],
        "outside_zero_ok": 1
      },
      "arc_gen_60pct_compatible_holdout": {
        "ok": 135,
        "total": 135,
        "bad_first10": [],
        "outside_zero_ok": 135
      },
      "arc_gen_compatible_count": 225,
      "arc_gen_

In [10]:
with zipfile.ZipFile(SUBMISSION_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_PATH, arcname=f"{TASK_ID}.onnx")

print("Wrote:", SUBMISSION_PATH)
print("Zip contents:", zipfile.ZipFile(SUBMISSION_PATH).namelist())
assert zipfile.ZipFile(SUBMISSION_PATH).namelist() == [f"{TASK_ID}.onnx"]

Wrote: /kaggle/working/submission.zip
Zip contents: ['task202.onnx']
